<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">📚 RAG como Herramienta</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 5 — Cuando la herramienta del agente es su propia base de conocimiento</p>
</div>

Las herramientas de los notebooks anteriores ejecutaban cálculos. Este notebook usa una herramienta distinta: una que <strong>busca información en un documento propio</strong> antes de responder — el patrón conocido como <strong>RAG</strong> (<em>Retrieval-Augmented Generation</em>, generación aumentada por recuperación). Es la forma más directa de que un agente responda con información específica y actualizada que el LLM nunca vio durante su entrenamiento.

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#problema-conocimiento">El problema: información que el LLM no conoce</a></li>
<li><a href="#construir-base">Construir una base de conocimiento</a></li>
<li><a href="#rag-como-tool">RAG envuelto como herramienta del agente</a></li>
<li><a href="#cierre">Cierre y próximos pasos</a></li>
</ol>
</div>

<a id="problema-conocimiento"></a>

## <span style="color:#F97066;">El problema: información que el LLM no conoce</span>

Un LLM solo "sabe" lo que vio durante su entrenamiento. Ante una pregunta sobre información específica, privada o posterior a ese entrenamiento — la política de garantía de una empresa ficticia, por ejemplo — no tiene forma de responder correctamente, aunque puede sonar convincente al inventar una respuesta plausible.

In [1]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GOOGLE_API_KEY"))

pregunta = "¿Cuántos meses de garantía tienen los auriculares NimbusTech Aria-7 y por qué?"
respuesta_sin_rag = llm.invoke(pregunta)
print(respuesta_sin_rag.text)

Para poder decirte exactamente cuántos meses de garantía tienen los auriculares **NimbusTech Aria-7** y por qué, **necesito más contexto**, ya que la duración de la garantía depende de varios factores que no se especifican solo con el nombre del modelo.

Aquí te explico por qué y cómo puedes averiguarlo:

1. **El país de compra:** Las leyes de protección al consumidor varían según el país. Por ejemplo, en la Unión Europea la garantía legal suele ser de 3 años (para productos comprados a partir de 2022), en México o Colombia suele ser de 1 año, y en Estados Unidos lo común es 1 año de garantía limitada del fabricante.
2. **La política del fabricante (NimbusTech):** Muchas marcas ofrecen un año estándar de garantía internacional, pero algunas extienden este periodo si registras el producto en su web oficial, o lo reducen si se trata de ciertos componentes.
3. **El tipo de vendedor:** No es lo mismo comprarlo directamente en la tienda oficial de NimbusTech que en un distribuidor autorizad

NimbusTech y el modelo Aria-7 son ficticios — se inventaron para este notebook, así que el LLM nunca pudo haber visto esta información en su entrenamiento. Aun así, responde con una explicación extensa y razonable en apariencia, basada en generalidades del mercado, en lugar de admitir que no conoce esa marca específica. Esto es exactamente el riesgo que RAG busca reducir: en vez de dejar que el LLM "adivine" con generalidades, se le da acceso directo al documento correcto.

<a id="construir-base"></a>

## <span style="color:#F97066;">Construir una base de conocimiento</span>

El patrón RAG tiene tres piezas: dividir el documento en fragmentos manejables, convertir cada fragmento en un vector numérico (*embedding*) que captura su significado, y guardar esos vectores en una base que permita buscar por similitud semántica.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

documento_politica_garantia = """
Política de garantía de NimbusTech (edición 2026).

Todos los productos NimbusTech tienen una garantía estándar de 18 meses desde la fecha de
compra, cubriendo defectos de fabricación.

Los auriculares NimbusTech modelo Aria-7 tienen una garantía extendida de 30 meses, debido
a que su batería usa una química de estado sólido con mayor vida útil.

La garantía no cubre daños por agua salvo en los modelos con certificación IP68, como el
parlante NimbusTech Orbe-2.

Para reclamar la garantía, el cliente debe presentar la factura original y el número de
serie del producto en cualquier centro de servicio NimbusTech.
"""

# 1. Dividir el documento en fragmentos pequeños y con algo de solapamiento entre ellos
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
fragmentos = splitter.split_text(documento_politica_garantia)

print(f"El documento se dividió en {len(fragmentos)} fragmentos.")
for i, fragmento in enumerate(fragmentos):
    print(f"\n--- Fragmento {i} ---\n{fragmento.strip()}")

El documento se dividió en 4 fragmentos.

--- Fragmento 0 ---
Política de garantía de NimbusTech (edición 2026).

Todos los productos NimbusTech tienen una garantía estándar de 18 meses desde la fecha de
compra, cubriendo defectos de fabricación.

--- Fragmento 1 ---
Los auriculares NimbusTech modelo Aria-7 tienen una garantía extendida de 30 meses, debido
a que su batería usa una química de estado sólido con mayor vida útil.

--- Fragmento 2 ---
La garantía no cubre daños por agua salvo en los modelos con certificación IP68, como el
parlante NimbusTech Orbe-2.

--- Fragmento 3 ---
Para reclamar la garantía, el cliente debe presentar la factura original y el número de
serie del producto en cualquier centro de servicio NimbusTech.


<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
Dividir el documento importa: pasarle el documento completo al LLM en cada pregunta sería posible aquí porque es corto, pero con documentos grandes (manuales de cientos de páginas) no cabría en la ventana de contexto del modelo, y sería mucho más costoso en tokens. Dividir en fragmentos permite recuperar solo los 2 o 3 más relevantes para cada pregunta.
</div>

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 1. Modelo de embeddings: convierte cada fragmento de texto en un vector numérico
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=os.getenv("GOOGLE_API_KEY"))

# 2. Base vectorial en memoria: guarda los vectores y permite buscar por similitud semántica
vectorstore = InMemoryVectorStore.from_texts(fragmentos, embedding=embeddings)

print("Base de conocimiento vectorial construida correctamente.")

Base de conocimiento vectorial construida correctamente.


In [4]:
# Búsqueda directa, sin agente: los fragmentos más parecidos semánticamente a la pregunta
resultados = vectorstore.similarity_search("¿cuánto dura la garantía de los auriculares?", k=2)
for doc in resultados:
    print(doc.page_content.strip())
    print("---")

Los auriculares NimbusTech modelo Aria-7 tienen una garantía extendida de 30 meses, debido
a que su batería usa una química de estado sólido con mayor vida útil.
---
Política de garantía de NimbusTech (edición 2026).

Todos los productos NimbusTech tienen una garantía estándar de 18 meses desde la fecha de
compra, cubriendo defectos de fabricación.
---


La búsqueda encontró el fragmento correcto aunque la pregunta no usa las mismas palabras exactas del documento ("dura la garantía" en vez de "garantía extendida de 30 meses") — la similitud es semántica, no una simple coincidencia de palabras clave.

<a id="rag-como-tool"></a>

## <span style="color:#F97066;">RAG envuelto como herramienta del agente</span>

Con la base de conocimiento lista, el último paso es envolver esa búsqueda en una función <code>@tool</code>, exactamente como cualquier otra herramienta de los notebooks anteriores. Para el agente, buscar en un documento no es distinto de calcular un área o convertir una moneda: es una herramienta más entre las disponibles.

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def buscar_politica_garantia(consulta: str) -> str:
    """Busca información sobre la política de garantía de productos NimbusTech.

    Args:
        consulta: La pregunta o tema a buscar en la política de garantía.
    """
    documentos = vectorstore.similarity_search(consulta, k=2)
    return "\n---\n".join(doc.page_content.strip() for doc in documentos)


agente = create_agent(
    model=llm,
    tools=[buscar_politica_garantia],
    system_prompt=(
        "Responde preguntas sobre NimbusTech usando la herramienta de búsqueda "
        "cuando la pregunta lo requiera, en vez de responder de memoria."
    ),
)

print("Agente con herramienta RAG construido correctamente.")

Agente con herramienta RAG construido correctamente.


In [6]:
resultado = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})

for mensaje in resultado["messages"]:
    if getattr(mensaje, "tool_calls", None):
        for llamada in mensaje.tool_calls:
            print(f"→ Herramienta invocada: {llamada['name']}({llamada['args']})")

print("\nRespuesta:\n")
print(resultado["messages"][-1].text)

→ Herramienta invocada: buscar_politica_garantia({'consulta': 'auriculares NimbusTech Aria-7 garantia meses por que'})

Respuesta:

Los auriculares NimbusTech Aria-7 tienen una garantía extendida de **30 meses**, debido a que su batería utiliza una química de estado sólido que ofrece una mayor vida útil en comparación con los modelos estándar.


Esta vez la respuesta es correcta y verificable: 30 meses, por la química de estado sólido de la batería — exactamente lo que dice el documento, en lugar de una generalización inventada del mercado. La diferencia frente a la primera respuesta de este notebook no es que el LLM "razone mejor": es que ahora tiene acceso al texto correcto antes de responder.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
RAG reduce el riesgo de que el modelo invente información, pero no lo elimina: si la base de conocimiento no tiene la respuesta, o si la búsqueda recupera el fragmento equivocado, el LLM puede seguir generando una respuesta incorrecta a partir de lo que sí recuperó. RAG ancla la respuesta en un texto real, pero la calidad final depende de qué tan buena sea esa recuperación.
</div>

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎯 Cierre y próximos pasos</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen</strong><br>
Este notebook mostró RAG como una herramienta más del agente:

- Sin acceso a información específica, un LLM puede responder con generalidades convincentes pero incorrectas.
- RAG divide un documento en fragmentos, los convierte en vectores (*embeddings*) y los guarda en una base que permite buscar por similitud semántica, no por coincidencia exacta de palabras.
- Envolver esa búsqueda en una función <code>@tool</code> integra RAG al mismo mecanismo de herramientas ya visto — para el agente, buscar información no es distinto de cualquier otra acción.
- RAG reduce, pero no elimina, el riesgo de respuestas incorrectas: depende de la calidad de la recuperación.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>➡️ Continúe con</strong>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>6-multiagentes.ipynb</code> — varios agentes especializados coordinados por un agente supervisor.</li>
</ul>
</div>